# Engineering Review Console — SQLite Warehouse

**Ticket:** DATA-214  
**Owner mode:** Anda bertindak sebagai engineer yang menyiapkan delivery untuk pull-request review.

Notebook ini hanya review console. Implementasi tetap di `schema.sql`, `queries.sql`, dan `src/warehouse.py`; final build harus dapat direproduksi lewat CLI.


## 1. Repository and environment check
Konfirmasi working directory, runtime, dan import path sebelum menganalisis kegagalan lain.


In [ ]:
from pathlib import Path
import sys, sqlite3, importlib, os

candidates = [Path.cwd(), Path.cwd().parent]
project_root = None
for candidate in candidates:
    if (candidate / "src" / "io_utils.py").exists() and (candidate / "src" / "warehouse.py").exists():
        project_root = candidate.resolve()
        break

if project_root is None:
    raise RuntimeError(
        "Project root tidak ditemukan. Buka notebook dari folder project utama yang berisi src/, data/, schema.sql, dan queries.sql. "
        f"Current working directory: {Path.cwd()}"
    )

if Path.cwd().resolve() != project_root:
    os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.io_utils import read_json

print("Python", sys.version.split()[0], "SQLite", sqlite3.sqlite_version)
print("project_root", project_root)
print("cwd", Path.cwd())
assert Path("src/warehouse.py").exists(), "File src/warehouse.py tidak ditemukan dari root project"
print("Environment: READY")

## 2. Input profiling
Inspeksi shape dan volume input read-only. Identifikasi risiko relasional; jangan memperbaiki source data di notebook.


In [ ]:
users=read_json("data/processed/users.json")
products=read_json("data/processed/products.json")
transactions=read_json("data/processed/transactions.json")
for n,x in [("users",users),("products",products),("transactions",transactions)]:print(n,type(x).__name__,len(x),x[0])

### Design checkpoint
Catat grain, candidate keys, nullability, dan risiko relasional pada `workpapers/SCHEMA_CANVAS.md` sebelum mengubah schema.


## 3. Load current implementation
Reload module agar review menggunakan perubahan source terbaru dan bukan state kernel lama.


In [ ]:
import src.warehouse as warehouse
importlib.reload(warehouse)
print("Loaded from",warehouse.__file__)

## 4. Connection contract
Buktikan FK enforcement aktif pada koneksi yang dipakai pipeline. Deklarasi FK tanpa enforcement bukan kontrol yang efektif.


In [ ]:
from pathlib import Path
db=Path("data/output/notebook_app.db");db.unlink(missing_ok=True)
try:
 conn=warehouse.connect_db(db)
 print("foreign_keys",conn.execute("PRAGMA foreign_keys").fetchone()[0])
except NotImplementedError as e:print("TODO",e)

## 5. Schema inspection
Build schema dan inspeksi table inventory serta foreign keys sebagai evidence untuk design review.


In [ ]:
try:
 warehouse.create_schema(conn)
 print("tables",conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())
 print("transaction FKs",conn.execute("PRAGMA foreign_key_list(transactions)").fetchall())
except (NotImplementedError,sqlite3.Error) as e:print(type(e).__name__,e)

### Reviewer checkpoint
Siapkan penjelasan: constraint apa yang melindungi risiko bisnis, bukan hanya sintaks DDL yang digunakan.


## 6. Master data load
Muat referenced entities sebelum transaction facts. Verifikasi counts segera setelah step ini untuk membatasi area diagnosis.


In [ ]:
try:
 print("loaded",warehouse.load_master(conn,users,products))
 print("users",conn.execute("SELECT COUNT(*) FROM users").fetchone()[0])
 print("products",conn.execute("SELECT COUNT(*) FROM products").fetchone()[0])
except (NotImplementedError,sqlite3.Error) as e:print(type(e).__name__,e)

## 7. Transaction load and rejection audit
Proses facts dengan failure isolation. Setiap reject harus terlihat, sedangkan row valid lainnya tetap committed.


In [ ]:
try:
 loaded,rejected=warehouse.load_transactions(conn,transactions)
 print("loaded",loaded,"rejected",len(rejected))
 for row in rejected:print(row)
except (NotImplementedError,sqlite3.Error) as e:print(type(e).__name__,e)

**Acceptance signal:** 22 accepted dan 3 rejected. Jika berbeda, triage PRAGMA, FK DDL, load order, dan transaction boundary.


## 8. Volume reconciliation
Bandingkan final table counts dengan acceptance criteria. Count membuktikan volume, belum membuktikan metric logic.


In [ ]:
try:print(warehouse.query_counts(conn))
except NotImplementedError as e:print("TODO",e)

## 9. Revenue query review
Verifikasi join condition, cardinality, dan grain sebelum menyetujui revenue. Dokumentasikan sample trace di `workpapers/JOIN_TRACE.md`.


In [ ]:
try:print("Revenue",warehouse.query_revenue(conn))
except NotImplementedError as e:print("TODO",e)

## 10. Product ranking review
Inspeksi aggregation grain, deterministic ordering, dan limit.


In [ ]:
try:
 for row in warehouse.query_top_products(conn):print(row)
except NotImplementedError as e:print("TODO",e)

**Acceptance signal:** P006 Webcam 19; P008 Notebook 2; P007 Desk Lamp 1.


## 11. Failure-control demonstration
Tunjukkan risiko koneksi SQLite default: constraint yang dideklarasikan tidak otomatis enforced pada setiap connection.


In [ ]:
unsafe=sqlite3.connect(":memory:")
print("default foreign_keys",unsafe.execute("PRAGMA foreign_keys").fetchone()[0])
unsafe.close()
print("Explain: kenapa nilai 0 berbahaya?")

## 12. Clean build reconciliation
Jalankan production entry point dan final validator. Evidence notebook harus sama dengan clean CLI build.


In [ ]:
import subprocess
for cmd in [[sys.executable, "run_pipeline.py"], [sys.executable, "validate_delivery.py"]]:
    result = subprocess.run(cmd, text=True, capture_output=True)
    print("$", " ".join(cmd))
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    assert result.returncode == 0, f"Command failed: {' '.join(cmd)}"


## Pull-request sign-off
1. hapus generated DB lama;
2. restart kernel dan Run All;
3. jalankan clean CLI build;
4. isi workpapers;
5. serahkan command, output, dan artifacts untuk review.


## Handoff questions
Reviewer harus dapat menemukan jawaban berikut dari source dan evidence:
- kontrol apa yang menegakkan referential integrity;
- bagaimana failure diisolasi dan dicatat;
- bagaimana metric grain diverifikasi;
- bagaimana rebuild dan recovery dilakukan.
